# Pipeline 2 - Donor Upgrade Propensity / Ask Amount Predictor

Generated: 2026-04-07T11:53:52.837068Z


## 1) Problem Framing

**Business question:** Predict likely next donation amount so fundraising asks can be personalized.

**Who cares:** nonprofit leadership, program staff, and/or fundraising staff depending on the pipeline domain.

**Why it matters:** this model turns operational, donor, or outreach data into a decision-support signal for a resource-constrained safehouse nonprofit.

**Predictive vs. explanatory goal:** this notebook includes both. The predictive model is evaluated on unseen data and is used for deployment-oriented scoring. The explanatory or relationship model is included to identify which variables appear most connected to the target and to support business interpretation. We do not treat predictive accuracy as causal proof.

**Success metrics:** classification pipelines use accuracy, F1, and ROC AUC where appropriate. Regression/forecasting pipelines use MAE, RMSE, and R-squared. The notebook also compares against a simple baseline so the results can be interpreted honestly.


## Notebook Setup

Shared imports and helper functions are defined once here so the later rubric sections can focus on the pipeline-specific code for this business problem.


In [6]:
import warnings

warnings.filterwarnings("ignore")

import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline

# Ensure the shared pipeline modules are importable when running from anywhere.
# (Repo-relative, so it works after pushing to GitHub.)
cwd = Path.cwd().resolve()
repo_root = cwd.parent if cwd.name == "ml-pipelines" else cwd
if (repo_root / "ml-pipelines").exists():
    ml_pipelines_dir = (repo_root / "ml-pipelines").resolve()
    if str(ml_pipelines_dir) not in sys.path:
        sys.path.insert(0, str(ml_pipelines_dir))

from data_prep import (
    as_bool,
    compare_profiles,
    dataset_profile,
    export_predictions_json,
    fill_numeric_median,
    get_project_paths,
    load_df,
    numeric,
    prep,
    quick_eda,
    time_split,
)
from modeling import (
    CandidateResult,
    ablate_feature_groups_one_by_one_cv,
    cv_evaluate_candidate,
    cv_results_table,
    eval_regression,
    evaluate_candidate,
    read_json_if_exists,
    results_table,
    select_simplest_within_delta,
    select_simplest_within_delta_cv,
    should_export_by_guardrail,
    top_features,
    tune_model_randomized,
    within_tolerance_rate,
    write_ablation_report_json,
    write_run_report_json,
)

paths = get_project_paths()
print("Data dir:", paths.data_dir)
print("Output dir:", paths.out_dir)


def print_business_takeaway(text):
    print("\nBusiness takeaway:")
    print(text)


Data dir: /Users/xenorex/Downloads/lighthouse_csv_v7
Output dir: /Users/xenorex/RiderProjects/INTEX2/output/ml-predictions


## 2) Data Acquisition, Preparation, and Exploration

The pipeline reads the provided CSV files from `data/raw/`, performs joins and feature engineering in code, handles missing values reproducibly, and prints an EDA summary before modeling. The EDA step checks row counts, missingness, target distributions, feature summaries, and key categorical distributions.


In [7]:
supporters = load_df("supporters")
donations = load_df("donations")
donations["donation_date"] = pd.to_datetime(donations["donation_date"], errors="coerce")
mon = donations[donations["donation_type"] == "Monetary"].copy()
mon["amount"] = numeric(mon["amount"])
mon = mon.dropna(subset=["donation_date","supporter_id"]).sort_values(["supporter_id","donation_date"]).copy()
mon["is_recurring_bool"] = as_bool(mon["is_recurring"])
mon["next_amount"] = mon.groupby("supporter_id")["amount"].shift(-1)
mon["next_date"] = mon.groupby("supporter_id")["donation_date"].shift(-1)
mon["days_to_next"] = (mon["next_date"] - mon["donation_date"]).dt.days
df = mon[(mon["next_amount"].notna()) & (mon["days_to_next"].between(1, 365))].copy()
df = df.merge(supporters[["supporter_id","supporter_type","relationship_type","region","country","acquisition_channel"]], on="supporter_id", how="left")
df["donations_so_far"] = df.groupby("supporter_id").cumcount() + 1
df["prev_amount"] = df.groupby("supporter_id")["amount"].shift(1).fillna(df["amount"])
df["prev_date"] = df.groupby("supporter_id")["donation_date"].shift(1)
df["recency_days"] = (df["donation_date"] - df["prev_date"]).dt.days.fillna(9999).clip(lower=0)
df["lifetime_amount_before"] = df.groupby("supporter_id")["amount"].cumsum() - df["amount"]
df["avg_amount_before"] = (df["lifetime_amount_before"] / (df["donations_so_far"] - 1).replace(0, np.nan)).fillna(df["amount"])
df["max_amount_before"] = df.groupby("supporter_id")["amount"].cummax()
df["amount_trend_vs_avg"] = (df["amount"] / df["avg_amount_before"].replace(0, np.nan)).fillna(1.0)
df["days_to_next"] = df["days_to_next"].clip(lower=1)
cat_cols = ["supporter_type","relationship_type","region","country","acquisition_channel","campaign_name","channel_source"]
num_cols = ["amount","prev_amount","recency_days","donations_so_far","is_recurring_bool"]
df[cat_cols] = df[cat_cols].fillna("Unknown")
fill_numeric_median(df, num_cols + ["next_amount"])
train, test = time_split(df, "donation_date")
print("Rows:", len(df), "Train:", len(train), "Test:", len(test))
quick_eda(df, "Donor upgrade modeling table", target_col="next_amount", numeric_cols=num_cols + ["next_amount"], categorical_cols=cat_cols)


Rows: 150 Train: 112 Test: 38

EDA: Donor upgrade modeling table
Shape: (150, 30)

Top missing-value rates:
referral_post_id          0.820000
prev_date                 0.293333
donation_id               0.000000
days_to_next              0.000000
max_amount_before         0.000000
avg_amount_before         0.000000
lifetime_amount_before    0.000000
recency_days              0.000000
prev_amount               0.000000
donations_so_far          0.000000

Target distribution / summary: next_amount
count     150.000000
mean     1034.636133
std       816.240510
min       250.000000
25%       469.575000
50%       800.665000
75%      1333.155000
max      6481.540000
Saved EDA plot: /Users/xenorex/RiderProjects/INTEX2/output/ml-predictions/eda-plots/donor_upgrade_modeling_table_next_amount_distribution.png

Numeric feature summary:
                      mean       std    min      50%      max
amount            1084.726   834.009  250.0  898.920  6481.54
prev_amount       1056.295   800.145  

## 3) Modeling and Feature Selection

The feature set is selected from fields that would be available at the decision point whenever possible. Predictive models use ensembles or tuned tree-based models when they improve out-of-sample performance. Explanatory models use simpler linear or logistic models when interpretability matters.


In [8]:
features = cat_cols + num_cols

# Keep an interpretable baseline for explanation.
explanatory = Pipeline([("pre", prep(cat_cols, num_cols)), ("model", LinearRegression())])

# Candidate pool for resource-aware selection.
candidates = {
    "LinearRegression": explanatory,
    "GradientBoosting": Pipeline([("pre", prep(cat_cols, num_cols)), ("model", GradientBoostingRegressor(random_state=42))]),
    "RandomForestSmall": Pipeline([("pre", prep(cat_cols, num_cols)), ("model", RandomForestRegressor(n_estimators=200, random_state=42, min_samples_leaf=3))]),
}



## 4) Evaluation and Selection

The notebook uses a train/test or time-based holdout split depending on the business problem. Where appropriate, it also uses compact cross-validation, model comparison tables, and lightweight randomized tuning. Metrics are interpreted in business terms rather than treated as abstract statistics.


In [9]:
# Strict holdout discipline:
# - Use CV on `train` for selection + tuning + ablation.
# - Touch `test` once at the end for final reporting.
X_train, y_train = train[features], train["next_amount"]
X_holdout, y_holdout = test[features], test["next_amount"]

baseline_value = float(np.median(y_train))
print("Baseline (median):", {"baseline_value": baseline_value, **eval_regression(y_holdout, np.repeat(baseline_value, len(y_holdout)))})

# Drift profile (feature space)
current_profile = dataset_profile(train[features], categorical_cols=cat_cols, numeric_cols=num_cols)
prev_report_path = paths.reports_dir / "donor-upgrade-propensity_run_report.json"
prev_report = read_json_if_exists(prev_report_path)
prev_profile = (prev_report or {}).get("data_profile")
drift = compare_profiles(current_profile, prev_profile)
print("Drift status:", drift.get("status"))

# 1) CV tournament
cv_results = [cv_evaluate_candidate(name, est, X_train, y_train, task="regression") for name, est in candidates.items()]
cv_tbl = cv_results_table(cv_results).sort_values(by=["mae_mean", "fit_s_mean"], ascending=True)
print("CV tournament (train only):")
print(cv_tbl.to_string(index=False))

best_mae = float(cv_tbl["mae_mean"].min())
delta = 0.05 * best_mae
selected_cv = select_simplest_within_delta_cv(cv_results, primary_metric="mae", delta=delta, higher_is_better=False)
base_selected = selected_cv.estimator
base_selected_name = selected_cv.name
print("Pre-tuning selected family:", base_selected_name)

# 2) Tune selected family
param_space = {}
if base_selected_name.startswith("RandomForest"):
    param_space = {
        "model__n_estimators": [100, 150, 200, 300],
        "model__max_depth": [None, 3, 5, 8],
        "model__min_samples_leaf": [1, 3, 5, 8],
    }
elif base_selected_name.startswith("GradientBoosting"):
    param_space = {
        "model__n_estimators": [100, 150, 250],
        "model__learning_rate": [0.03, 0.05, 0.1],
        "model__max_depth": [2, 3, 4],
    }

if param_space:
    tune = tune_model_randomized(
        base_selected_name,
        base_selected,
        X_train,
        y_train,
        param_distributions=param_space,
        task="regression",
        cv_metric="neg_mean_absolute_error",
        n_iter=12,
    )
    tuned_estimator = tune.best_estimator
    tuned_name = f"{base_selected_name}+tuned"
    tuned_params = tune.best_params
else:
    tuned_estimator = base_selected
    tuned_name = base_selected_name
    tuned_params = {}

# 3) Post-tune CV check
post_cv = cv_evaluate_candidate(tuned_name, tuned_estimator, X_train, y_train, task="regression")

# 4) Ablation (CV) with group-aware single-column groups
feature_groups = [[c] for c in features]
ablation_tbl = ablate_feature_groups_one_by_one_cv(
    base_name=tuned_name,
    estimator=tuned_estimator,
    X=X_train,
    y=y_train,
    task="regression",
    feature_groups=feature_groups,
    primary_metric="mae",
    higher_is_better=False,
)
print("Ablation top removals (CV):")
print(ablation_tbl.head(15).to_string(index=False))

ablation_path = paths.reports_dir / "donor-upgrade-propensity_ablation_report.json"
write_ablation_report_json(ablation_path, {"pipeline": "donor-upgrade-propensity", "table": ablation_tbl.to_dict(orient="records")})
print("Wrote ablation report:", ablation_path)

# 5) Final holdout evaluation
final_model = tuned_estimator
final_model.fit(X_train, y_train)
holdout_pred = np.maximum(0, final_model.predict(X_holdout))
holdout_metrics = eval_regression(y_holdout, holdout_pred)
holdout_metrics["within_25pct"] = within_tolerance_rate(y_holdout, holdout_pred, tolerance=0.25, relative=True)

# Guardrail export: block if MAE worsens by >5% relative vs previous holdout
allow_export, guardrail = should_export_by_guardrail(
    previous_report=prev_report,
    current_holdout={"mae": float(holdout_metrics["mae"])},
    primary_metric="mae",
    min_delta_allowed=0.05 * float((prev_report or {}).get("holdout", {}).get("mae", holdout_metrics["mae"])),
    higher_is_better=False,
)

run_report = {
    "pipeline": "donor-upgrade-propensity",
    "primary_metric": "mae",
    "selected_family": base_selected_name,
    "tuned_name": tuned_name,
    "tuned_params": tuned_params,
    "cv_table": cv_tbl.to_dict(orient="records"),
    "post_tune_cv": post_cv.metrics_mean,
    "holdout": holdout_metrics,
    "guardrail": guardrail,
    "data_profile": current_profile,
    "drift": drift,
    "notes": {"split": "time_split(donation_date)"},
}
write_run_report_json(prev_report_path, run_report)
print("Wrote run report:", prev_report_path)

selected_model = final_model
selected_model_name = tuned_name



Baseline (median): {'baseline_value': 773.2650000000001, 'mae': 623.181052631579, 'rmse': 848.9008586114845, 'r2': -0.2306385365553476}
Drift status: ok
CV tournament (train only):
            model  fit_s_mean  predict_s_mean   mae_mean   rmse_mean   r2_mean   mae_std   rmse_std   r2_std
RandomForestSmall    0.091388        0.008046 534.348543  776.125869 -0.013663 53.147142 251.038295 0.117428
 GradientBoosting    0.030299        0.008666 611.858456  873.508137 -0.298541 77.724637 259.622232 0.186725
 LinearRegression    0.006615        0.001659 774.698786 1052.835739 -1.099468 11.138555 141.753356 0.889548
Pre-tuning selected family: RandomForestSmall
Ablation top removals (CV):
      dropped_group  n_cols_dropped   mae_base  mae_after  delta_helpful  fit_s_mean_after  predict_s_mean_after
       recency_days               1 532.302259 520.320997      11.981262          0.062895              0.006411
     supporter_type               1 532.302259 530.141760       2.160500          0

## 5) Causal and Relationship Analysis

The relationship analysis section highlights important features and discusses whether those relationships make sense for the organization. These findings are observational: they can guide hypotheses and strategy, but they are not automatically causal. Any sensitive resident-care decision must remain human-reviewed.


In [10]:
print("Top predictive features:")
print(top_features(selected_model).to_string(index=False))
print("Top explanatory relationships:")
print(top_features(explanatory).to_string(index=False))
print_business_takeaway("Use suggested ask tiers as fundraising guidance, not an automated rule. Validate large ask increases with relationship context.")


Top predictive features:
                             feature  importance
cat__acquisition_channel_SocialMedia    0.285150
                         num__amount    0.148127
                    num__prev_amount    0.119585
              num__is_recurring_bool    0.089683
                   num__recency_days    0.071634
          cat__channel_source_Direct    0.044388
               num__donations_so_far    0.033996
   cat__supporter_type_MonetaryDonor    0.027932
                   cat__region_Luzon    0.027063
        cat__channel_source_Campaign    0.023270
Top explanatory relationships:
Empty DataFrame
Columns: []
Index: []

Business takeaway:
Use suggested ask tiers as fundraising guidance, not an automated rule. Validate large ask increases with relationship context.


## 6) Deployment Notes

The final scoring step exports JSON to `output/ml-predictions/`. These files match the API import contract used by `POST /api/ml/import?replace=true` and can be viewed in the deployed staff portal under `/app/ml` or the ML action center.


In [11]:
latest = mon.groupby("supporter_id").tail(1).copy()
latest = latest.merge(supporters[["supporter_id","supporter_type","relationship_type","region","country","acquisition_channel"]], on="supporter_id", how="left")
history = mon.groupby("supporter_id").agg(
    donations_so_far=("donation_id","count"),
    prev_amount=("amount", lambda s: s.iloc[-2] if len(s) >= 2 else s.iloc[-1]),
    prev_date=("donation_date", lambda s: s.iloc[-2] if len(s) >= 2 else pd.NaT),
    lifetime_amount_before=("amount","sum"),
    avg_amount_before=("amount","mean"),
    max_amount_before=("amount","max"),
).reset_index()
latest = latest.merge(history, on="supporter_id", how="left")
latest["recency_days"] = (latest["donation_date"] - latest["prev_date"]).dt.days.fillna(9999).clip(lower=0)
latest["is_recurring_bool"] = as_bool(latest["is_recurring"])
latest["amount_trend_vs_avg"] = (latest["amount"] / latest["avg_amount_before"].replace(0, np.nan)).fillna(1.0)
latest["days_to_next"] = latest["recency_days"]
latest[cat_cols] = latest[cat_cols].fillna("Unknown")
fill_numeric_median(latest, num_cols)
latest["predicted_next_amount"] = np.maximum(0, selected_model.predict(latest[features]))
latest["upgrade_ratio"] = (latest["predicted_next_amount"] / latest["amount"].replace(0, np.nan)).fillna(0)
latest["ask_tier"] = pd.cut(latest["upgrade_ratio"], [-np.inf, 1.05, 1.25, 1.50, np.inf], labels=["Maintain","Small Upgrade","Upgrade","Major Upgrade"])
export_predictions_json(
    "donor_upgrade_next_amount",
    "Supporter",
    latest[["supporter_id","predicted_next_amount","ask_tier","amount","upgrade_ratio","recency_days","donations_so_far","supporter_type","acquisition_channel"]].assign(export_model=selected_model_name),
    "supporter_id",
    "predicted_next_amount",
    "ask_tier",
)


Wrote 57 predictions: /Users/xenorex/RiderProjects/INTEX2/output/ml-predictions/donor_upgrade_next_amount.json


PosixPath('/Users/xenorex/RiderProjects/INTEX2/output/ml-predictions/donor_upgrade_next_amount.json')